# Cross-Validation Hyperparameter Selection Analysis

Compares hyperparameter selection via **single train/val split** (BASELINE) vs **CV-averaged scoring** (CROSS_VAL) across basins and modes.

**Key questions:**
1. Does CV pick different hyperparameters than single-split?
2. Does CV improve test-period generalization?
3. Is the effect consistent across basins and resolutions (Daily, MTS-1D, MTS-1H)?

**CV config:** 4-fold rolling window, 2-year intervals, 1-year validation, starting October.

In [1]:
import sys
sys.path.insert(0, "../..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from UCB_training.UCB_eval import load_test_metrics, pairwise_pct_change, threshold_filter

%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

In [2]:
# ── Utility Functions ─────────────────────────────────────────────

def load_hparams_csv(path):
    """Load a hyperparams CSV (best params or gridsearch results)."""
    path = Path(path)
    if not path.exists():
        print(f"  Not found: {path}")
        return None
    return pd.read_csv(path)


def compare_best_hparams(baseline_df, cv_df, label_a='BASELINE', label_b='CROSS_VAL'):
    """Side-by-side comparison of best hyperparams from two runs.
    Both inputs should be DataFrames with model_type column.
    """
    rows = []
    for model_type in ['no_physics', 'physics']:
        a = baseline_df.query(f"model_type == '{model_type}'").iloc[0]
        b = cv_df.query(f"model_type == '{model_type}'").iloc[0]
        
        # Find common hyperparam columns (exclude metrics and model_type)
        metric_cols = {'NSE_1D', 'NSE_1H', 'model_type', 'learning_rate'}
        hp_cols = [c for c in a.index if c not in metric_cols and c in b.index]
        
        for col in hp_cols:
            val_a = a[col]
            val_b = b[col]
            changed = '***' if str(val_a) != str(val_b) else ''
            rows.append({
                'model': model_type,
                'param': col,
                label_a: val_a,
                label_b: val_b,
                'changed': changed,
            })
        
        # Add NSE metrics
        for m in ['NSE_1D', 'NSE_1H']:
            if m in a.index and m in b.index:
                val_a = a[m]
                val_b = b[m]
                diff = val_b - val_a if pd.notna(val_a) and pd.notna(val_b) else None
                rows.append({
                    'model': model_type,
                    'param': m,
                    label_a: round(val_a, 4) if pd.notna(val_a) else None,
                    label_b: round(val_b, 4) if pd.notna(val_b) else None,
                    'changed': f'{diff:+.4f}' if diff is not None else '',
                })
    
    return pd.DataFrame(rows)


def load_gridsearch_pair(shared_dir, label):
    """Load no-physics and physics gridsearch CSVs for a given label.
    Returns (df_no_phys, df_phys) or (None, None).
    """
    hp_dir = Path(shared_dir) / 'hyperparams'
    basin = shared_dir.parent.name
    mode = shared_dir.name.replace('_shared', '')
    
    no_phys_path = hp_dir / f"{basin}_{mode}_{label}_no_physics_gridsearch.csv"
    phys_path = hp_dir / f"{basin}_{mode}_{label}_physics_gridsearch.csv"
    
    df_no = load_hparams_csv(no_phys_path)
    df_phys = load_hparams_csv(phys_path)
    return df_no, df_phys


def plot_gridsearch_comparison(df_a, df_b, metric='NSE_1H',
                               label_a='BASELINE', label_b='CROSS_VAL',
                               title=''):
    """Bar chart comparing grid search results between two runs."""
    if df_a is None or df_b is None:
        print(f"  Skipping plot — missing data")
        return
    
    # Sort both by the same param combination for alignment
    id_cols = ['hidden_size', 'seq_length_1D', 'seq_length_1H', 'batch_size']
    id_cols = [c for c in id_cols if c in df_a.columns and c in df_b.columns]
    
    if not id_cols:
        print("  No common ID columns found")
        return
    
    df_a = df_a.sort_values(id_cols).reset_index(drop=True)
    df_b = df_b.sort_values(id_cols).reset_index(drop=True)
    
    # Create combo labels
    labels = df_a[id_cols].apply(
        lambda r: '\n'.join(f"{c.split('_')[-1]}={int(r[c])}" for c in id_cols),
        axis=1
    )
    
    x = np.arange(len(labels))
    width = 0.35
    
    fig, ax = plt.subplots(figsize=(max(8, len(labels) * 1.5), 4))
    ax.bar(x - width/2, df_a[metric], width, label=label_a, alpha=0.8)
    ax.bar(x + width/2, df_b[metric], width, label=label_b, alpha=0.8)
    
    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=8)
    ax.set_ylabel(metric)
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.show()


print("Utility functions loaded.")

Utility functions loaded.


---
## 1. Calpella Daily: BASELINE vs CVTEST_V2

Original experiment — single-split vs 3-fold CV for daily-only models.

In [3]:
CALP_DAILY = Path("../../outputs/calpella/daily")

CALP_DAILY_BASELINE = CALP_DAILY / "BASELINE_20250815T000000Z"
CALP_DAILY_CV = CALP_DAILY / "CVTEST_V2_20250815T000000Z"

if CALP_DAILY_BASELINE.exists() and CALP_DAILY_CV.exists():
    baseline_m = load_test_metrics(CALP_DAILY_BASELINE, "calpella")
    cv_m = load_test_metrics(CALP_DAILY_CV, "calpella")
    
    comp = pairwise_pct_change(baseline_m, cv_m, models=["LSTM", "PILSTM"])
    comp = comp.rename(columns={"EXPERIMENTAL": "CVTEST_V2"})
    
    filtered = threshold_filter(
        pairwise_pct_change(baseline_m, cv_m, models=["LSTM", "PILSTM"]),
        threshold=5, always_keep=["NSE"]
    ).rename(columns={"EXPERIMENTAL": "CVTEST_V2"})
    
    print(f"Metrics with >5% change (or NSE):")
    display(filtered.round(3))
else:
    print("Calpella daily BASELINE or CVTEST_V2 not found.")

Metrics with >5% change (or NSE):


,Metric,Model,BASELINE,CVTEST_V2
0,NSE,LSTM,0.802,0.804
1,NSE,PILSTM,0.841,0.848
6,KGE,LSTM,0.723,0.820
8,Alpha-NSE,LSTM,0.750,0.875
13,Beta-NSE,PILSTM,-0.045,-0.024
16,FHV,LSTM,-23.218,-10.334
18,FMS,LSTM,-29.641,-18.615
19,FMS,PILSTM,-12.444,-13.980
20,FLV,LSTM,-78.330,41.415
21,FLV,PILSTM,54.654,13.352


### 1a. Hyperparameter Comparison — Calpella Daily

In [4]:
CALP_DAILY_SHARED = Path("../../outputs/calpella/daily_shared/hyperparams")

bl_hp = load_hparams_csv(CALP_DAILY_SHARED / "calpella_daily_BASELINE_hyperparams.csv")
cv_hp = load_hparams_csv(CALP_DAILY_SHARED / "calpella_daily_CVTEST_V2_hyperparams.csv")

if bl_hp is not None and cv_hp is not None:
    hp_comp = compare_best_hparams(bl_hp, cv_hp, 'BASELINE', 'CVTEST_V2')
    display(hp_comp)
else:
    # Try ES_PATIENCE as alternative
    es_hp = load_hparams_csv(CALP_DAILY_SHARED / "calpella_daily_ES_PATIENCE_hyperparams.csv")
    if bl_hp is not None and es_hp is not None:
        print("CVTEST_V2 hparams not found, showing BASELINE vs ES_PATIENCE:")
        hp_comp = compare_best_hparams(bl_hp, es_hp, 'BASELINE', 'ES_PATIENCE')
        display(hp_comp)

  Not found: ../../outputs/calpella/daily_shared/hyperparams/calpella_daily_CVTEST_V2_hyperparams.csv
  Not found: ../../outputs/calpella/daily_shared/hyperparams/calpella_daily_ES_PATIENCE_hyperparams.csv


---
## 2. MTS: BASELINE vs CROSS_VAL — Hyperparameter Shifts

Compare what hyperparameters CV selects vs single-split across all basins.

In [5]:
MTS_BASINS = {
    'calpella': Path('../../outputs/calpella/mts_shared'),
    'guerneville': Path('../../outputs/guerneville/mts_shared'),
    'hopland': Path('../../outputs/hopland/mts_shared'),
    'warm_springs': Path('../../outputs/warm_springs/mts_shared'),
}

# Check what hyperparams CSVs exist
for basin, shared in MTS_BASINS.items():
    hp_dir = shared / 'hyperparams'
    if hp_dir.exists():
        csvs = sorted(hp_dir.glob('*.csv'))
        labels = set()
        for c in csvs:
            # Extract label from filename: basin_mode_LABEL_hyperparams.csv
            name = c.stem
            if 'gridsearch' not in name and 'hyperparams' in name:
                parts = name.split('_')
                # Find the label between mode and 'hyperparams'
                hp_idx = parts.index('hyperparams') if 'hyperparams' in parts else -1
                if hp_idx > 2:
                    label = '_'.join(parts[2:hp_idx])
                    labels.add(label)
        print(f"{basin:15s}: {', '.join(sorted(labels)) if labels else 'no hyperparams'}")
    else:
        print(f"{basin:15s}: no shared dir")

calpella       : BASELINE, CROSS_VAL
guerneville    : BASELINE, CROSS_VAL
hopland        : BASELINE
warm_springs   : mts_BASELINE


### 2a. Best Hyperparameters — BASELINE vs CROSS_VAL

In [6]:
for basin, shared in MTS_BASINS.items():
    hp_dir = shared / 'hyperparams'
    mode = 'mts'
    
    bl_path = hp_dir / f"{basin}_{mode}_BASELINE_hyperparams.csv"
    cv_path = hp_dir / f"{basin}_{mode}_CROSS_VAL_hyperparams.csv"
    
    bl = load_hparams_csv(bl_path)
    cv = load_hparams_csv(cv_path)
    
    if bl is not None and cv is not None:
        print(f"\n{'='*60}")
        print(f"{basin.upper()} MTS — BASELINE vs CROSS_VAL")
        print(f"{'='*60}")
        display(compare_best_hparams(bl, cv))
    elif bl is not None:
        print(f"\n{basin}: BASELINE found, CROSS_VAL not yet available")
    elif cv is not None:
        print(f"\n{basin}: CROSS_VAL found, no BASELINE for comparison")


CALPELLA MTS — BASELINE vs CROSS_VAL


,model,param,BASELINE,CROSS_VAL,changed
0,no_physics,hidden_size,256.0000,128.0000,***
1,no_physics,seq_length_1D,90.0000,90.0000,
2,no_physics,seq_length_1H,336.0000,336.0000,
3,no_physics,num_layers,1.0000,1.0000,
4,no_physics,epochs,16.0000,32.0000,***
5,no_physics,batch_size,64.0000,128.0000,***
6,no_physics,output_dropout,0.4000,0.4000,
7,no_physics,NSE_1D,0.8138,0.7885,-0.0253
8,no_physics,NSE_1H,0.7649,0.7906,+0.0257
9,physics,hidden_size,64.0000,64.0000,



GUERNEVILLE MTS — BASELINE vs CROSS_VAL


,model,param,BASELINE,CROSS_VAL,changed
0,no_physics,hidden_size,256.0000,256.0000,
1,no_physics,seq_length_1D,90.0000,90.0000,
2,no_physics,seq_length_1H,168.0000,168.0000,
3,no_physics,num_layers,1.0000,1.0000,
4,no_physics,epochs,48.0000,32.0000,***
5,no_physics,batch_size,64.0000,64.0000,
6,no_physics,output_dropout,0.4000,0.4000,
7,no_physics,NSE_1D,0.8136,0.7687,-0.0449
8,no_physics,NSE_1H,0.7946,0.8777,+0.0832
9,physics,hidden_size,128.0000,256.0000,***


  Not found: ../../outputs/hopland/mts_shared/hyperparams/hopland_mts_CROSS_VAL_hyperparams.csv

hopland: BASELINE found, CROSS_VAL not yet available
  Not found: ../../outputs/warm_springs/mts_shared/hyperparams/warm_springs_mts_CROSS_VAL_hyperparams.csv

warm_springs: BASELINE found, CROSS_VAL not yet available


### 2b. Full Grid Search Results — Score Distributions

In [7]:
for basin, shared in MTS_BASINS.items():
    bl_no, bl_phys = load_gridsearch_pair(shared, 'BASELINE')
    cv_no, cv_phys = load_gridsearch_pair(shared, 'CROSS_VAL')
    
    if cv_no is not None and bl_no is not None:
        print(f"\n{'='*60}")
        print(f"{basin.upper()} MTS — Grid Search Score Comparison")
        print(f"{'='*60}")
        
        for metric in ['NSE_1H', 'NSE_1D']:
            if metric in cv_no.columns and metric in bl_no.columns:
                plot_gridsearch_comparison(
                    bl_no, cv_no, metric=metric,
                    title=f"{basin.title()} No-Physics — {metric}"
                )
            if cv_phys is not None and bl_phys is not None:
                if metric in cv_phys.columns and metric in bl_phys.columns:
                    plot_gridsearch_comparison(
                        bl_phys, cv_phys, metric=metric,
                        title=f"{basin.title()} Physics — {metric}"
                    )


CALPELLA MTS — Grid Search Score Comparison


ValueError: shape mismatch: objects cannot be broadcast to a single shape.  Mismatch is between arg 0 with shape (108,) and arg 1 with shape (72,).

---
## 3. CV Fold-Level Analysis

How consistent is each hyperparameter combo across folds? High fold variance suggests overfitting to a specific period.

In [ ]:
for basin, shared in MTS_BASINS.items():
    runs_dir = shared / 'runs'
    cv_grid_dirs = sorted(runs_dir.glob('CROSS_VAL_*_grid_*')) if runs_dir.exists() else []
    
    if not cv_grid_dirs:
        continue
    
    print(f"\n{'='*60}")
    print(f"{basin.upper()} — CV Fold Variance")
    print(f"{'='*60}")
    
    for grid_dir in cv_grid_dirs:
        # Find cv_summary.csv
        summaries = list(grid_dir.rglob('cv_summary.csv'))
        if not summaries:
            continue
        
        cv_sum = pd.read_csv(summaries[0])
        grid_label = grid_dir.name.split('_grid_')[-1]
        
        print(f"\n  Grid {grid_label}:")
        
        for metric in ['NSE_1D', 'NSE_1H']:
            if metric in cv_sum.columns:
                vals = cv_sum[metric].dropna()
                if len(vals) > 1:
                    print(f"    {metric}: mean={vals.mean():.4f}, "
                          f"std={vals.std():.4f}, "
                          f"range=[{vals.min():.4f}, {vals.max():.4f}]")

---
## 4. Cross-Basin Summary

Aggregate: does CV consistently pick different/better params across basins?

In [ ]:
summary_rows = []

for basin, shared in MTS_BASINS.items():
    hp_dir = shared / 'hyperparams'
    mode = 'mts'
    
    bl = load_hparams_csv(hp_dir / f"{basin}_{mode}_BASELINE_hyperparams.csv")
    cv = load_hparams_csv(hp_dir / f"{basin}_{mode}_CROSS_VAL_hyperparams.csv")
    
    if bl is None or cv is None:
        continue
    
    for model_type in ['no_physics', 'physics']:
        bl_row = bl.query(f"model_type == '{model_type}'").iloc[0]
        cv_row = cv.query(f"model_type == '{model_type}'").iloc[0]
        
        row = {'basin': basin, 'model': model_type}
        
        for m in ['NSE_1D', 'NSE_1H']:
            if m in bl_row.index and m in cv_row.index:
                bl_val = bl_row[m]
                cv_val = cv_row[m]
                row[f'BL_{m}'] = round(bl_val, 4) if pd.notna(bl_val) else None
                row[f'CV_{m}'] = round(cv_val, 4) if pd.notna(cv_val) else None
                row[f'd_{m}'] = round(cv_val - bl_val, 4) if pd.notna(bl_val) and pd.notna(cv_val) else None
        
        # Flag param changes
        param_changes = []
        for p in ['hidden_size', 'seq_length_1H', 'batch_size']:
            if p in bl_row.index and p in cv_row.index:
                if str(bl_row[p]) != str(cv_row[p]):
                    param_changes.append(f"{p.split('_')[-1]}: {int(bl_row[p])}→{int(cv_row[p])}")
        row['param_changes'] = ', '.join(param_changes) if param_changes else 'same'
        
        summary_rows.append(row)

if summary_rows:
    summary_df = pd.DataFrame(summary_rows)
    display(summary_df)
else:
    print("Not enough data for cross-basin summary yet.")

---
## 5. Findings & Observations

*To be filled as results come in.*

**Running observations:**
- Calpella MTS: CV trades ~2-3% daily NSE for ~2.5% hourly NSE improvement
- CV tends to shift toward longer hourly sequences (168→336) and larger batches (64→128)
- BASELINE no-physics picked h=256 which is NOT in the new trimmed grid — re-run will differ